# OpenSSL: keys and file formats

Run cells from top to bottom. Code cells create files or print evidence; answer cells are where you write your observations. Modify only cells marked `TODO`. Generated keys are random, so their values differ between runs.

In [ ]:
!openssl version

## Generate the working files
Run this cell once. It creates RSA and EC keys in PEM and DER. Encryption, signatures and key exchange follow in `05_rsa_from_scratch.ipynb` and `06_ec_from_scratch.ipynb`.

In [ ]:
!openssl genpkey -algorithm RSA -outform PEM -pkeyopt rsa_keygen_bits:2048 -pkeyopt rsa_keygen_pubexp:3 -out RSA_2048_priv.pem
!openssl pkey -inform PEM -outform DER -in RSA_2048_priv.pem -out RSA_2048_priv.der
!openssl pkey -pubout -inform PEM -outform PEM -in RSA_2048_priv.pem -out RSA_2048_pub.pem
!openssl pkey -pubin -pubout -inform PEM -outform DER -in RSA_2048_pub.pem -out RSA_2048_pub.der

!openssl genpkey -algorithm EC -outform PEM -pkeyopt ec_paramgen_curve:P-256 -pkeyopt ec_param_enc:named_curve -out EC_256_priv.pem
!openssl pkey -inform PEM -outform DER -in EC_256_priv.pem -out EC_256_priv.der
!openssl pkey -pubout -inform PEM -outform PEM -in EC_256_priv.pem -out EC_256_pub.pem
!openssl pkey -pubin -pubout -inform PEM -outform DER -in EC_256_pub.pem -out EC_256_pub.der

## Tasks
Useful commands:
- `openssl asn1parse -i -inform DER -in FILE` prints a DER ASN.1 structure.
- `openssl pkey -text -noout -in FILE` prints key details. Add `-pubin` for a public key.

### Task 1: Convert PEM to DER
Decode `RSA_2048_pub.pem` and verify that it produces exactly `RSA_2048_pub.der`. Explain the purpose of the PEM header/footer and Base64 encoding.

In [ ]:
import base64

pem_lines = open("RSA_2048_pub.pem").read().splitlines()
der_from_pem = base64.b64decode("".join(pem_lines[1:-1]))
der_file = open("RSA_2048_pub.der", "rb").read()

print(der_from_pem == der_file)
assert der_from_pem == der_file

### Answers
- PEM header/footer:
- Base64 purpose:

### Task 2: Read a DER RSA public key
Run the cells in order. Start with the ASN.1 structure; inspect the raw bytes only in the final cell. Modify only the cell marked `TODO`.

**a)** Identify the outer `SEQUENCE`, the RSA algorithm OID, and the `BIT STRING`. Record the `BIT STRING` offset.

**b)** Replace `??` with that offset and inspect the nested structure. What two values does it contain?

**c)** In the final cell, locate the DER bytes for the modulus and public exponent. Identify their tag, length, and value bytes. Which ASN.1 type stores both numbers?

In [ ]:
!openssl asn1parse -i -inform DER -in RSA_2048_pub.der

In [ ]:
# TODO: replace ?? with the BIT STRING offset
!openssl asn1parse -i -inform DER -in RSA_2048_pub.der -strparse ??

In [ ]:
der = open("RSA_2048_pub.der", "rb").read()
for offset in range(0, len(der), 16):
    chunk = der[offset:offset + 16]
    print(f"{offset:03}: {chunk.hex(' ')}")

### Answers
- Outer SEQUENCE:
- RSA algorithm OID:
- BIT STRING offset:
- Nested values:
- Modulus TLV bytes:
- Exponent TLV bytes:
- ASN.1 type for both numbers:

### Task 3: Inspect RSA and EC key details
Run the cells below and record the values shown by OpenSSL.

**a)** Record the first and last 8 hexadecimal digits of RSA `n`, `e`, `d`, `p`, and `q` (notebook 04 decodes the full values automatically). How many bits does `n` have? Explain the leading `00:` byte – compare with the INTEGER encoding on the slides (128 → `00 80`).

**b)** Record the EC private key, public point, and curve. The public point is stored as `04 ‖ x ‖ y` (`04` = uncompressed point). How many bytes do x and y have, and why?

**c)** In the `asn1parse` outputs of `RSA_2048_priv.der` and `EC_256_priv.der`, find the RSA `INTEGER`s, the EC private key (`OCTET STRING`), the curve identifier and the public point (`BIT STRING`).

In [ ]:
!openssl pkey -inform PEM -text -noout -in RSA_2048_priv.pem
!openssl pkey -pubin -inform PEM -text -noout -in RSA_2048_pub.pem

In [ ]:
!openssl pkey -inform PEM -text -noout -in EC_256_priv.pem
!openssl pkey -pubin -inform PEM -text -noout -in EC_256_pub.pem

In [ ]:
!openssl asn1parse -i -inform DER -in RSA_2048_priv.der

In [ ]:
!openssl asn1parse -i -inform DER -in EC_256_priv.der

### Answers
- RSA values and bit length:
- Leading `00:`:
- EC private key, public point, and curve:
- Size of x and y:
- RSA ASN.1 locations:
- EC ASN.1 locations: